In [ ]:
import os
import sys
from pathlib import Path

HOURS_TO_DELIVER = 1000 # 50
CTE_CORRECTION = 1000
max_time_per_truck = HOURS_TO_DELIVER * CTE_CORRECTION # 
DATA_VERSION = "data_version_1"
HOURS=4
MIN=60
SEC = 60
running_time = HOURS*MIN*SEC # sec

# This finds the project root regardless of the machine or folder execution
try:
    # If running as a script
    current_file = Path(__file__).resolve()
    project_root = current_file.parent.parent
except NameError:
    # If running in a Jupyter Notebook
    # We look for the 'core' folder to identify the root
    path = Path(os.getcwd())
    while path.parent != path:
        if (path / "core").exists():
            project_root = path
            break
        path = path.parent
    else:
        project_root = Path(os.getcwd())

os.chdir(project_root)


if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
from core.utils.data_loader import MDVRPDataLoader
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp



data_path = os.path.join(project_root, "data", DATA_VERSION)
print(f"Checking data at: {data_path}")
if os.path.exists(data_path):
    print("Data folder found!")





def solve_vrp_and_save_results(data_version):
    # 1. Configuration & Data Loading
    loader = MDVRPDataLoader(data_dir=data_version)
    data_dict = loader.load_data()
    
    time_matrix = data_dict["time_matrix"].numpy()
    num_vehicles = len(data_dict["trucks"])
    depot_index = 0  # Assuming all trucks start from the first node (Depot)

    # 2. Routing Index Manager & Model
    manager = pywrapcp.RoutingIndexManager(len(time_matrix), num_vehicles, depot_index)
    routing = pywrapcp.RoutingModel(manager)

    # 3. Transit Callback (using CTE_CORRECTION as scaling factor for high precision)
    def time_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return int(time_matrix[from_node][to_node] * CTE_CORRECTION)

    transit_callback_index = routing.RegisterTransitCallback(time_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # 4. Add Time Dimension (to track visit times per customer)
    routing.AddDimension(
        transit_callback_index,
        0,                   # No slack (waiting time)
        max_time_per_truck,  # Updated from 100000 to 24000
        True,                # Start cumulative time at zero
        "Time"
    )
    time_dimension = routing.GetDimensionOrDie("Time")

    # 5. Search Parameters (Metaheuristics for exact/near-exact shortest path)
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = running_time # Adjust based on complexity

    # 6. Solve the Problem
    solution = routing.SolveWithParameters(search_parameters)

    # 7. Process and Save Results
    if solution:
        # Paths for the report
        output_folder = os.path.join("data", data_version)
        output_file = os.path.join(output_folder, "routing_results.txt")
        os.makedirs(output_folder, exist_ok=True)

        total_combined_time = solution.ObjectiveValue() / 1000
        
        # Open file to write the report
        with open(output_file, "w", encoding="utf-8") as f:
            header = f"=== FINAL ROUTING REPORT ({data_version}) ===\n"
            header += f"Total Combined Time of all Trucks: {total_combined_time:.2f} hours\n\n"
            print(header) # Print to console
            f.write(header)

            for vehicle_id in range(num_vehicles):
                index = routing.Start(vehicle_id)
                truck_id = data_dict['trucks'][vehicle_id].id
                truck_output = f"--- TRUCK {truck_id} ---\n"
                
                route_steps = []
                while not routing.IsEnd(index):
                    node_idx = manager.IndexToNode(index)
                    node_id = data_dict['idx_to_node'][node_idx]
                    arrival_time = solution.Value(time_dimension.CumulVar(index)) / 1000
                    
                    route_steps.append(f"{node_id} (Visited at: {arrival_time:.2f}h)")
                    index = solution.Value(routing.NextVar(index))
                
                # Arrival back at depot
                final_node_idx = manager.IndexToNode(index)
                final_node_id = data_dict['idx_to_node'][final_node_idx]
                total_route_time = solution.Value(time_dimension.CumulVar(index)) / 1000
                route_steps.append(f"{final_node_id} (Back at: {total_route_time:.2f}h)")
                
                truck_output += " -> ".join(route_steps) + "\n"
                truck_output += f"Total Route Time: {total_route_time:.2f} hours\n\n"
                
                print(truck_output) # Print to console
                f.write(truck_output)
        
        print(f"Results successfully saved to: {output_file}")
    else:
        print("No solution found!")

if __name__ == "__main__":
    solve_vrp_and_save_results(DATA_VERSION)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

# 1. Load the data (assuming you are in the project root)
depot_df = pd.read_csv(os.path.join("data", DATA_VERSION, "selected_depot.csv"))
customer_df = pd.read_csv(os.path.join("data", DATA_VERSION, "selected_customers.csv"))

# 2. Create the plot
plt.figure(figsize=(12, 8))

# Plot Customers
plt.scatter(customer_df['longitude'], customer_df['latitude'], 
            c='blue', label='Customers', alpha=0.6, s=20)

for i, row in depot_df.iterrows():
    plt.annotate(row['id_depot'], (row['longitude'], row['latitude']), xytext=(5, 5), textcoords='offset points', fontweight='bold')


# Plot Depots
plt.scatter(depot_df['longitude'], depot_df['latitude'], 
            c='red', label='Depots', marker='D', s=100, edgecolors='black')

# Add labels and styling
plt.title(f"Node Distribution - {DATA_VERSION}", fontsize=15)
plt.xlabel("Longitude", fontsize=12)
plt.ylabel("Latitude", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

output_folder = os.path.join("data", DATA_VERSION)
plot_file = os.path.join(output_folder, DATA_VERSION + "_node_distribution.png")
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
print(f"Plot successfully saved to: {plot_file}")

plt.show()